# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [11]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, update_display, Markdown

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

In [15]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

openai = OpenAI()
llama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")



In [3]:
def streaming_response(ai:OpenAI, model:str,sys_prompt:str, question:str):
    stream = ai.chat.completions.create(
        model = model,
        messages = [
        {"role":"system", "content":sys_prompt},
        {"role":"user", "content":question}
        ],
        stream=True
    )
    response = ""
    display(Markdown(""), display_id="response_stream")
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id="response_stream")

In [4]:
# here is the question; type over this to ask something new

sys_prompt = """
    you're a helpful assistant
"""

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""



In [5]:
# Get gpt-4o-mini to answer, with streaming
streaming_response(openai, MODEL_GPT, sys_prompt, question)

The code snippet you provided uses the `yield from` statement in Python along with a set comprehension to yield values from a collection. Let's break it down:

1. **Set Comprehension**: 
   - `{book.get("author") for book in books if book.get("author")}` is a set comprehension that iterates through a collection called `books`.
   - For each `book` in `books`, it attempts to get the value associated with the key `"author"` using `book.get("author")`.
   - The `if book.get("author")` part filters out any `book` entries that do not have a valid author value (i.e., it only includes non-`None` or non-empty values). 
   - The result of this set comprehension is a set containing unique author values extracted from all books that have an author.

2. **`yield from` Statement**:
   - The `yield from` expression is used in a generator function to yield all values from the iterable that follows it. In this case, it's yielding all unique authors from the set created by the comprehension.
   - Using `yield from` allows the generator to yield values one by one, which is efficient in terms of memory usage since it doesn't require creating a full list of results before yielding them.

### What this code does:
- The code is likely part of a generator function that, when called, will yield each unique author from a list of `books` where each book is expected to be a dictionary that may or may not have an "author" key.
- Only authors that are both present and valid (non-null) will be included in the yields.

### Example Context:
Assuming you have a list of book dictionaries like this:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},  # Duplicate author
    {"title": "Book 4"},                          # No author key
    {"title": "Book 5", "author": ""}            # Empty author
]
```

The code will yield:
- "Author A"
- "Author B"

It will not yield any `None` or empty strings, resulting in each author being provided only once. 

This is useful for scenarios where you want to process or list authors from a collection of books without duplicates and without any invalid entries.

In [16]:
# Get Llama 3.2 to answer
streaming_response(llama, MODEL_LLAMA, sys_prompt, question)

**Breaking Down the Code**

The given code snippet is a Python expression that utilizes the `yield from` syntax to create an iterator. Here's a breakdown of what it does:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

* `yield from`: This keyword indicates that the expression inside should be treated as a `genexpr` (generator expression), which is a compact syntax for creating generators.
* `{...for ...if ...}`: This is a dictionary comprehension generator expression. It iterates over the `book` variable, applying a filter (`if book.get("author")`) and yielding a value (`book.get("author")`) for each matching element.
* `{...for ...if ...}` is equivalent to `for book in {book get("author") for book in books if book.get("author")}`. But it's much more readable.

**What the Code Does**

In summary, the code retrieves the authors from a list of books, filtering out books without an author.

Here's an equivalent, more explicit implementation for comparison:

```python
authors = []
for book in books:
    if book.get("author"):
        authors.append(book.get("author"))

yields from authors
```

**Why `yield from`?**

Using `yield from` offers several advantages:

* **Readability**: The code is more expressive and easier to understand than the explicit implementation.
* **Efficiency**: The `yield from` syntax does not create an intermediate list of authors, which would result in O(n) space usage. Instead, it yields author values one by one, achieving O(1) memory usage while still producing an iterator.
* **Flexibility**: The `yield from` expression can be used in functions and with other iterable contexts, making it a powerful tool for working with generators and iterators.

When to use `yield from`:

* When working with complex iterables, generator expressions, or list comprehensions.
* When you need to create an iterator from a more complex iterable with multiple loops or conditions.
* When optimizing memory usage, as `yield from` avoids creating intermediate data structures.